In [1]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/raw/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")

In [7]:
print(df.shape)

(286467, 79)


In [8]:
print(df.columns.tolist())

[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count', ' URG Flag 

In [3]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,1266342,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,22,1319353,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,22,160,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,22,1303488,41,42,2728,6634,456,0,66.536585,110.129945,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,35396,77,1,2,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [9]:
# strip whitespaces
df.columns = df.columns.str.strip()
print(df['Label'].value_counts())


Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64


In [10]:
features = ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
            'Total Length of Fwd Packets', 'SYN Flag Count', 'ACK Flag Count']

df.groupby('Label')[features].mean()

,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,SYN Flag Count,ACK Flag Count
Label,,,,,,
BENIGN,1.197957e+07,6.533453,6.655237,522.910936,0.047155,0.278304
PortScan,8.282023e+04,1.017580,1.004958,1.089171,0.000000,0.000384


In [14]:
df['predicted-scan'] = (
    (df['Total Fwd Packets'] < 1.5) & 
    (df['Total Length of Fwd Packets'] < 1.5) & 
    (df['Flow Duration'] < 100000)
)

In [29]:
true_positives = ((df['Label'] == 'PortScan') & (df['predicted-scan'] == True)).sum()
false_positives = ((df['Label'] == 'BENIGN') & (df['predicted-scan'] == True)).sum()
true_negatives = ((df['Label'] == 'BENIGN') & (df['predicted-scan'] == False)).sum()
false_negatives = ((df['Label'] == 'PortScan') & (df['predicted-scan'] == False)).sum()
print(true_positives, false_positives, true_negatives, false_negatives)

79492 5788 121749 79438


In [33]:
detection_rate = true_positives / (true_positives + false_negatives)
print(f"{(detection_rate * 100).round()} % ")

50.0 % 


In [34]:
false_positive_rate = false_positives / (false_positives + true_negatives)
print(f"{(false_positive_rate * 100).round()} % ")

5.0 % 
